In [1]:
# ============================================================
# SIMPLE RAG USING CHROMADB + GEMINI
# NO LANGCHAIN
# ============================================================

# Install:
# pip install chromadb google-genai sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer
from google import genai

In [2]:
# ============================================================
# 1. GEMINI API SETUP
# ============================================================
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [3]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# Install all required libraries
!pip install -q chromadb sentence-transformers google-genai

# Import all required libraries
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
# ============================================================
# 2. LOAD EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# 3. CREATE CHROMADB CLIENT
# ============================================================

chroma_client = chromadb.PersistentClient(
    path="./rag_database"
)


# ============================================================
# 4. CREATE OR GET COLLECTION
# ============================================================

collection = chroma_client.get_or_create_collection(
    name="knowledge_base"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# ============================================================
# 5. SAMPLE DOCUMENTS
# ============================================================

documents = [

    """
    Generative AI refers to artificial intelligence systems
    that can generate new content such as text, images,
    audio, video and computer programs.
    """,

    """
    Retrieval-Augmented Generation, or RAG, combines
    information retrieval with a large language model.
    Relevant documents are retrieved from a knowledge base
    and supplied to the LLM as context.
    """,

    """
    ChromaDB is a vector database designed for storing
    embeddings and performing similarity search.
    It is commonly used in RAG applications.
    """,

    """
    Gemini is Google's family of multimodal large language
    models. Gemini models can understand and generate
    text and can work with several other modalities.
    """,

    """
    Embeddings convert text into numerical vectors.
    Texts with similar semantic meanings generally have
    vectors that are close to each other in embedding space.
    """
]


ids = [
    "doc1",
    "doc2",
    "doc3",
    "doc4",
    "doc5"
]


# ============================================================
# 6. CREATE EMBEDDINGS FOR DOCUMENTS
# ============================================================

document_embeddings = embedding_model.encode(
    documents
).tolist()


# ============================================================
# 7. STORE DOCUMENTS IN CHROMADB
# ============================================================

# Avoid adding duplicates every time the program runs

if collection.count() == 0:

    collection.add(

        ids=ids,

        documents=documents,

        embeddings=document_embeddings
    )

    print("Documents added to ChromaDB.")

else:

    print(
        "Documents already exist in ChromaDB."
    )

Documents added to ChromaDB.


In [7]:
# ============================================================
# 8. USER QUESTION
# ============================================================

question = input(
    "\nEnter your question: "
)


# ============================================================
# 9. CREATE QUERY EMBEDDING
# ============================================================

query_embedding = embedding_model.encode(
    question
).tolist()


# ============================================================
# 10. RETRIEVE SIMILAR DOCUMENTS
# ============================================================

results = collection.query(

    query_embeddings=[
        query_embedding
    ],

    n_results=3
)


retrieved_documents = (
    results["documents"][0]
)


# ============================================================
# 11. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print(
    "\nRetrieved Context:"
)

print(
    "=" * 60
)


for i, doc in enumerate(
    retrieved_documents,
    start=1
):

    print(
        f"\nDocument {i}:"
    )

    print(
        doc.strip()
    )


# ============================================================
# 12. COMBINE RETRIEVED DOCUMENTS
# ============================================================

context = "\n\n".join(
    retrieved_documents
)


# ============================================================
# 13. CREATE RAG PROMPT
# ============================================================

prompt = f"""
You are a helpful AI assistant.

Answer the user's question using only the
information provided in the context below.

If the answer is not available in the context,
say:

"I do not have enough information in the
knowledge base."

CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""


# ============================================================
# 14. SEND PROMPT TO GEMINI
# ============================================================

response = client.models.generate_content(

    model="gemini-2.5-flash",

    contents=prompt
)


# ============================================================
# 15. DISPLAY FINAL RAG ANSWER
# ============================================================

print(
    "\n" + "=" * 60
)

print(
    "RAG ANSWER"
)

print(
    "=" * 60
)

print(
    response.text
)


Enter your question: What is RAG?

Retrieved Context:

Document 1:
Retrieval-Augmented Generation, or RAG, combines
    information retrieval with a large language model.
    Relevant documents are retrieved from a knowledge base
    and supplied to the LLM as context.

Document 2:
ChromaDB is a vector database designed for storing
    embeddings and performing similarity search.
    It is commonly used in RAG applications.

Document 3:
Embeddings convert text into numerical vectors.
    Texts with similar semantic meanings generally have
    vectors that are close to each other in embedding space.

RAG ANSWER
Retrieval-Augmented Generation, or RAG, combines information retrieval with a large language model.
